# Native NeoOLAF DocRED **dev-only** streaming batch and evaluation — v6.2

This notebook keeps the frozen **v5.1 scientific configuration and evaluator**, but changes the batch orchestration so that:

- only records whose literal JSON key `"type"` equals `"dev"` are selected;
- the JSONL is scanned line by line instead of loaded into memory;
- no list containing all dev documents is constructed;
- at most `DOCUMENT_WORKERS` document jobs are submitted at once;
- the existing `LAYER_WORKERS` parallelism is retained inside each document;
- evaluation is rebuilt by streaming the per-document result files.

The intended workflow is:

1. leave `RUN_ALL_DEV_DOCUMENTS = False` and run the first five `type="dev"` records;
2. inspect the same strict DocRED relation/entity evaluation used previously;
3. change only `RUN_ALL_DEV_DOCUMENTS = True`;
4. rerun the configuration, preflight, and batch cells.

Completed runs under this notebook's batch root are resumed. Gold entities and relations are removed before each native NeoOLAF run and loaded only afterward for evaluation.

## Memory behavior

The parent process keeps only:

- the current JSONL line;
- at most `DOCUMENT_WORKERS` prepared/submitted jobs;
- small aggregate counters for 96 relation types and 13 layers.

Input/gold records and NeoOLAF artifacts are written to disk per document. The complete dev split is never materialized as a Python list or submitted to `ProcessPoolExecutor` all at once.

In [ ]:
from __future__ import annotations

import os
import sys
import multiprocessing as mp
from getpass import getpass
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "src/neoolaf").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the NeoOLAF repository.")


def first_existing_path(label: str, candidates: list[Path]) -> Path:
    checked = []
    for candidate in candidates:
        candidate = candidate.expanduser()
        candidate = candidate if candidate.is_absolute() else PROJECT_ROOT / candidate
        candidate = candidate.resolve()
        checked.append(candidate)
        if candidate.is_file():
            print(f"{label}={candidate}")
            return candidate
    raise FileNotFoundError(
        f"Could not find {label}. Checked:\n" + "\n".join(str(path) for path in checked)
    )


PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "examples/RAGTreeDatasets"
TOOLS_DIR = NOTEBOOK_DIR / "tools"
for path in [PROJECT_ROOT / "src", PROJECT_ROOT, TOOLS_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from docred_native_batch_v6_2_dev_streaming import (
    BatchRunConfig,
    aggregate_batch_results_streaming,
    count_exact_type_records,
    read_json,
    run_batch_streaming,
)

mp.freeze_support()
print("PROJECT_ROOT =", PROJECT_ROOT)

## 1. Exact `type="dev"` filter and one-variable smoke/full switch

In [ ]:
# Change only this variable after the five-document dev smoke run succeeds.
RUN_ALL_DEV_DOCUMENTS = False

# Used only while RUN_ALL_DEV_DOCUMENTS=False.
SMOKE_DEV_DOCUMENT_LIMIT = 5

# Index among matching type="dev" records. Keep 0 for the complete dev split.
START_DEV_INDEX = 0

# Frozen exact filter: it checks record["type"] only and does not fall back to "split".
EXACT_TYPE_VALUE = "dev"

DATASET_JSONL = first_existing_path(
    "DATASET_JSONL",
    [
        NOTEBOOK_DIR / "../../../ragtree/data/preprocessed/docred_causal.jsonl",
        PROJECT_ROOT.parent / "ragtree/data/preprocessed/docred_causal.jsonl",
        PROJECT_ROOT.parent / "RAGTree/data/preprocessed/docred_causal.jsonl",
        PROJECT_ROOT / "ragtree/data/preprocessed/docred_causal.jsonl",
    ],
)

# Separate root prevents collisions with the previous unfiltered five-document batch.
# Keep this same root when switching RUN_ALL_DEV_DOCUMENTS to True so smoke runs resume.
BATCH_ROOT = NOTEBOOK_DIR / "runs/docred_native_v5_1_dev_streaming"

print("RUN_ALL_DEV_DOCUMENTS =", RUN_ALL_DEV_DOCUMENTS)
print("DATASET_JSONL =", DATASET_JSONL)
print("BATCH_ROOT =", BATCH_ROOT)

## 2. Frozen v5.1 scientific configuration and parallel workers

In [ ]:
ONTOLOGY_PATH = NOTEBOOK_DIR / "ontology/docred_redocred_neoolaf_compatible.ttl"
ONTOLOGY_ORIGINAL = NOTEBOOK_DIR / "ontology/docred_redocred_original.ttl"
RELATION_CATALOG = NOTEBOOK_DIR / "ontology/docred_relation_catalog.json"
RELATION_ALIASES = NOTEBOOK_DIR / "ontology/docred_relation_aliases.json"
PROFILE_PATH = NOTEBOOK_DIR / "configs/docred_profile_native_ablation_v5.json"
GUIDANCE_PATH = NOTEBOOK_DIR / "configs/guidance_docred_native_ablation_v5.json"
TASK_GUIDANCE_PATH = NOTEBOOK_DIR / "configs/docred_task_guidance_v5_1_frozen.json"

OPENROUTER_HOST = "https://openrouter.ai/api/v1"
MODEL_NAME = "openai/gpt-oss-20b"
API_KEY = os.environ.get("OPENROUTER_API_KEY", "").strip().strip('"').strip("'")

# Parallel NeoOLAF documents. The streaming scheduler holds no more than this many jobs.
DOCUMENT_WORKERS = 4

# Existing v5.1 parallel relation-enrichment workers inside each document.
LAYER_WORKERS = 16

REASONING_EFFORT = "minimal"
MAX_TOKENS = 4096
REQUEST_TIMEOUT = 120

RESUME_COMPLETED = True
RETRY_FAILED_DOCUMENTS = True
DOCUMENT_ATTEMPTS = 2
RETRY_BACKOFF_SECONDS = 8.0
DOCUMENT_LAUNCH_STAGGER_SECONDS = 0.75
VERBOSE_DOCUMENTS = False
PROGRESS_EVERY = 1 if not RUN_ALL_DEV_DOCUMENTS else 10

print("Document workers / maximum pending documents:", DOCUMENT_WORKERS)
print("Layer workers per document:", LAYER_WORKERS)
print("Maximum theoretical concurrent Layer 2 requests:", DOCUMENT_WORKERS * LAYER_WORKERS)
print("API key available:", bool(API_KEY))

## 3. Streaming preflight and anti-leakage assertions

In [ ]:
required = [
    DATASET_JSONL,
    ONTOLOGY_PATH,
    ONTOLOGY_ORIGINAL,
    RELATION_CATALOG,
    RELATION_ALIASES,
    PROFILE_PATH,
    GUIDANCE_PATH,
    TASK_GUIDANCE_PATH,
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))

profile = read_json(PROFILE_PATH)
task_guidance = read_json(TASK_GUIDANCE_PATH)
catalog = read_json(RELATION_CATALOG)

# One streaming scan: counts exact type="dev" records without retaining documents.
corpus_scan = count_exact_type_records(
    DATASET_JSONL,
    required_type=EXACT_TYPE_VALUE,
    first_n_ids=5,
)
matching_records = int(corpus_scan["matching_records"])
available_after_start = max(0, matching_records - START_DEV_INDEX)
selected_count = (
    available_after_start
    if RUN_ALL_DEV_DOCUMENTS
    else min(SMOKE_DEV_DOCUMENT_LIMIT, available_after_start)
)

print("JSONL records:", corpus_scan["total_records"])
print('Records with exact key/value type="dev":', matching_records)
print("Selected this invocation:", selected_count)
print("First matching dev IDs:", corpus_scan["first_matching_ids"])
print("Ontology properties:", catalog["property_count"])
print("Allowed relation IDs:", len(task_guidance["allowed_relation_ids"]))
print("Frozen profile:", profile["profile_name"])

assert EXACT_TYPE_VALUE == "dev"
assert catalog["property_count"] == 96
assert len(task_guidance["allowed_relation_ids"]) == 96
assert profile["relations"]["allowed"] == []
assert profile["anti_cheating"]["direct_docred_extraction"] is False
assert profile["anti_cheating"]["source_entity_anchoring"] is False
assert profile["anti_cheating"]["post_run_relation_invention"] is False
assert profile["benchmark_projection"]["gold_available_to_pipeline"] is False
assert 3 <= DOCUMENT_WORKERS <= 5
assert selected_count > 0

display(Markdown(
    f"**Mode:** {'all exact type=dev documents' if RUN_ALL_DEV_DOCUMENTS else 'first five exact type=dev documents'}  \n"
    f"**Selected:** {selected_count} of {matching_records} dev documents  \n"
    f"**Parent memory bound:** at most {DOCUMENT_WORKERS} submitted document jobs  \n"
    f"**Resume:** {RESUME_COMPLETED}; keep the same batch root when switching to all dev documents."
))

## 4. Run native NeoOLAF with bounded document streaming

The launcher reads the JSONL iteratively and advances to another `type="dev"` record only after a document-worker slot is available. It does not prepare all dev records or submit all futures in advance.

Each selected document still receives its complete Layer 0–12 run, logs, prompts, raw responses, ontology artifacts, strict relation evaluation, entity evaluation, and cumulative layer analysis.

In [ ]:
RUN_PIPELINE = True

if RUN_PIPELINE:
    if not API_KEY:
        API_KEY = getpass("OpenRouter API key: ").strip().strip('"').strip("'")
    if not API_KEY:
        raise RuntimeError("No OpenRouter API key was provided.")

    config = BatchRunConfig(
        project_root=str(PROJECT_ROOT),
        ontology_path=str(ONTOLOGY_PATH),
        profile_path=str(PROFILE_PATH),
        guidance_path=str(GUIDANCE_PATH),
        relation_catalog_path=str(RELATION_CATALOG),
        relation_aliases_path=str(RELATION_ALIASES),
        model_name=MODEL_NAME,
        host=OPENROUTER_HOST,
        document_workers=DOCUMENT_WORKERS,
        layer_workers=LAYER_WORKERS,
        reasoning_effort=REASONING_EFFORT,
        max_tokens=MAX_TOKENS,
        request_timeout=REQUEST_TIMEOUT,
        resume_completed=RESUME_COMPLETED,
        retry_failed_documents=RETRY_FAILED_DOCUMENTS,
        document_attempts=DOCUMENT_ATTEMPTS,
        retry_backoff_seconds=RETRY_BACKOFF_SECONDS,
        launch_stagger_seconds=DOCUMENT_LAUNCH_STAGGER_SECONDS,
        verbose_documents=VERBOSE_DOCUMENTS,
        progress_every=PROGRESS_EVERY,
    )

    batch = run_batch_streaming(
        dataset_jsonl=DATASET_JSONL,
        task_guidance_path=TASK_GUIDANCE_PATH,
        batch_root=BATCH_ROOT,
        run_all_documents=RUN_ALL_DEV_DOCUMENTS,
        smoke_document_limit=SMOKE_DEV_DOCUMENT_LIMIT,
        start_index=START_DEV_INDEX,
        config=config,
        api_key=API_KEY,
        matching_record_count=matching_records,
        required_type=EXACT_TYPE_VALUE,
    )
    print("Dev-only bounded streaming execution and aggregate evaluation completed.")
else:
    batch = aggregate_batch_results_streaming(
        batch_root=BATCH_ROOT,
        relation_catalog_path=RELATION_CATALOG,
    )
    print("RUN_PIPELINE=False: aggregate evaluation rebuilt iteratively from saved document files.")

## 5. Aggregate relation and entity evaluation

In [ ]:
summary = batch["summary"]
display(pd.DataFrame([{
    "documents_requested": summary["documents_requested"],
    "documents_completed_or_resumed": summary["documents_completed_or_resumed"],
    "documents_failed": summary["documents_failed"],
    "relation_micro_precision": summary["micro_relation"]["precision"],
    "relation_micro_recall": summary["micro_relation"]["recall"],
    "relation_micro_f1": summary["micro_relation"]["f1"],
    "relation_macro_precision": summary["macro_relation"]["precision"],
    "relation_macro_recall": summary["macro_relation"]["recall"],
    "relation_macro_f1": summary["macro_relation"]["f1"],
    "entity_micro_precision": summary["micro_entity_inventory"]["precision"],
    "entity_micro_recall": summary["micro_entity_inventory"]["recall"],
    "entity_micro_f1": summary["micro_entity_inventory"]["f1"],
    "endpoint_micro_precision": summary["micro_relation_endpoint_inventory"]["precision"],
    "endpoint_micro_recall": summary["micro_relation_endpoint_inventory"]["recall"],
    "endpoint_micro_f1": summary["micro_relation_endpoint_inventory"]["f1"],
    "mean_pipeline_seconds": summary["mean_document_pipeline_seconds"],
    "median_pipeline_seconds": summary["median_document_pipeline_seconds"],
}]))

print("First-failure counts across completed documents:")
display(pd.DataFrame(
    sorted(summary["first_failure_counts"].items(), key=lambda item: (-item[1], item[0])),
    columns=["first_failure", "gold_relations"],
))

## 6. Per-document results (loaded from the streamed CSV)

In [ ]:
per_document_path = Path(batch["paths"]["per_document_csv"])
per_document = pd.read_csv(per_document_path) if per_document_path.is_file() else pd.DataFrame()

if not per_document.empty:
    print("Completed documents in CSV:", len(per_document))
    print("Twenty-five lowest relation-F1 documents:")
    display(per_document.sort_values(
        ["relation_f1", "relation_recall", "relation_precision"],
        ascending=[True, True, True],
    ).head(25))
    display(per_document[[
        "relation_precision", "relation_recall", "relation_f1",
        "entity_precision", "entity_recall", "entity_f1",
        "pipeline_seconds",
    ]].describe())
else:
    print("No completed documents.")

## 7. Per-relation micro evaluation

In [ ]:
per_relation = pd.DataFrame(batch["per_relation"])
if not per_relation.empty:
    display(per_relation.sort_values(
        ["gold", "f1", "recall"],
        ascending=[False, True, True],
    ))
else:
    print("No relation metrics available.")

## 8. Aggregate cumulative layer contribution

In [ ]:
cumulative = pd.DataFrame(batch["cumulative"])
if not cumulative.empty:
    display(cumulative[[
        "layer_index", "layer_name", "predicted", "gold",
        "true_positive", "false_positive", "false_negative",
        "precision", "recall", "f1",
    ]])
else:
    print("No cumulative layer metrics available.")

## 9. Failures, scheduler state, and saved outputs

In [ ]:
failed = pd.DataFrame(batch.get("failed_preview", []))
print("Failed documents:", summary["documents_failed"])
if not failed.empty:
    display(failed[[
        column for column in [
            "selection_index", "document_id", "title", "status",
            "error_type", "error", "run_dir",
        ]
        if column in failed.columns
    ]])

if "scheduler" in batch:
    display(pd.DataFrame([batch["scheduler"]]))

print("Batch output root:", BATCH_ROOT)
print("Selection stream:", BATCH_ROOT / "selection_documents.jsonl")
print("Selection manifest:", BATCH_ROOT / "selection_manifest.json")
print("Progress events:", BATCH_ROOT / "batch_events.jsonl")
print("Aggregate summary:", batch["paths"]["batch_summary"])
print("Per-document CSV:", batch["paths"]["per_document_csv"])
print("Per-relation CSV:", BATCH_ROOT / "aggregate_analysis/per_relation_metrics.csv")
print("Cumulative layer CSV:", BATCH_ROOT / "aggregate_analysis/cumulative_layer_micro_evaluation.csv")
print("Predictions JSONL:", batch["paths"]["predictions_jsonl"])
print("Failures JSONL:", batch["paths"]["failed_documents_jsonl"])

## 10. Launch the complete dev split

After the five-document dev smoke result is acceptable, change only:

```python
RUN_ALL_DEV_DOCUMENTS = True
```

Rerun the switch/configuration, preflight, and batch cells. Keep the same `BATCH_ROOT`. Completed smoke documents are detected by their scientific fingerprints and skipped.

The full run still reads all 106,924 JSONL lines to locate matching records, but it retains only the current line and at most `DOCUMENT_WORKERS` submitted document jobs. It does **not** create a list of every dev document in RAM.

For sustained HTTP 429 errors, reduce `DOCUMENT_WORKERS` from `4` to `3` before reducing `LAYER_WORKERS`.